# Greffe du classifieur

> Remplacement de la tête `linear` du modèle AAE et création d’un `Learner` fastai.


In [ ]:
#| default_exp graft


In [ ]:
#| export
"""Classifier grafting helpers for the fastai AAE."""


from pathlib import Path
from typing import Any

from torch import nn

from tell_me_why.config import HumanCenteredAAEConfig
from tell_me_why.source import build_human_centered_aae


def infer_out_features(module: nn.Module) -> int | None:
    """Infer the final output width of a classifier module when possible."""

    if hasattr(module, "out_features"):
        return int(module.out_features)
    for child in reversed(list(module.modules())):
        if child is module:
            continue
        if hasattr(child, "out_features"):
            return int(child.out_features)
    return None


def graft_classifier_to_human_aae(
    aae: nn.Module,
    classifier: nn.Module,
    *,
    classifier_attr: str = "linear",
    freeze_aae_body: bool = True,
    update_classes: bool = True,
) -> nn.Module:
    """Replace the Human-Centered AAE classification head with a supplied classifier."""

    setattr(aae, classifier_attr, classifier)

    if update_classes:
        out_features = infer_out_features(classifier)
        if out_features is not None:
            aae.classes = out_features

    if freeze_aae_body:
        prefix = f"{classifier_attr}."
        for name, parameter in aae.named_parameters():
            parameter.requires_grad_(name.startswith(prefix))

    return aae


def make_grafted_learner(
    dls: Any,
    classifier: nn.Module,
    *,
    aae: nn.Module | None = None,
    config: HumanCenteredAAEConfig | None = None,
    model_path: str | Path | None = None,
    pretrained_weights: str | None = None,
    loss_func: Any | None = None,
    metrics: list[Any] | None = None,
    freeze_aae_body: bool = True,
    load_strict: bool = False,
    **learner_kwargs: Any,
) -> Any:
    """Create a fastai `Learner` using the Human-Centered AAE plus a new head."""

    from fastai.vision.all import CrossEntropyLossFlat, Learner, accuracy

    model = aae if aae is not None else build_human_centered_aae(config, model_path=model_path)
    resolved_loss = loss_func or CrossEntropyLossFlat()
    resolved_metrics = metrics if metrics is not None else [accuracy]

    if pretrained_weights is not None:
        preload = Learner(dls, model, loss_func=resolved_loss, metrics=resolved_metrics)
        preload.load(pretrained_weights, strict=load_strict)

    model = graft_classifier_to_human_aae(
        model,
        classifier,
        freeze_aae_body=freeze_aae_body,
    )
    return Learner(
        dls,
        model,
        loss_func=resolved_loss,
        metrics=resolved_metrics,
        **learner_kwargs,
    )
